In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

from pyscf import gto

import sys

BASIS = "def2-svp"

In [ ]:
dict_ = {"molecule": [], "spin": {}, "charge": {}}

data_path = Path("./sets/dft-fitset")

for dir_path in data_path.glob("*"):
    # check if the path is a directory
    if not dir_path.is_dir():
        continue
    dataset = dir_path.stem
    print(f"Processing {dataset}")
    dict_[f"molecule_{dataset}"] = []

    # sort with the name of the molecule
    dir_path_list = list(dir_path.glob("*"))
    dir_path_list.sort(key=lambda x: x.name)
    for mol_path in dir_path_list:
        if not mol_path.is_dir():
            continue

        xyz_file = (mol_path / "mol.xyz").resolve().as_posix()
        try:
            mol = gto.M(
                atom=xyz_file,
                basis=BASIS,
                verbose=0,
                charge=0,
                spin=0,
                unit="B",
            )
            mol.build()
        except Exception as e:
            print(xyz_file)
            continue

        molecule = []
        for i_atom in mol._atom:
            molecule.append(
                [
                    i_atom[0],
                    i_atom[1][0],
                    i_atom[1][1],
                    i_atom[1][2],
                ]
            )

        i_name = mol_path.name
        dict_i_name = f"{dataset}-{i_name}"
        dict_[dict_i_name] = molecule
        dict_[f"molecule_{dataset}"].append(dict_i_name)
        dict_["molecule"].append(dict_i_name)
        dict_["charge"][dict_i_name] = mol.charge
        dict_["spin"][dict_i_name] = mol.spin

        if dataset != "NCIBLIND10":
            distance_str = f"molecule_{dataset}-{dict_i_name.split('-')[-1]}"
            if distance_str not in dict_:
                dict_[distance_str] = []
            dict_[distance_str].append(dict_i_name)
            distance_str = f"reaction-{dataset}-{dict_i_name.split('-')[-1]}"
            if distance_str not in dict_:
                dict_[distance_str] = {}

    reaction_pd = pd.read_csv(dir_path / "ref.csv")
    dict_[f"reaction-{dataset}"] = {}
    for i_reaction, reaction in enumerate(reaction_pd.itertuples()):
        systems = [reaction.dimer, reaction.monomerA, reaction.monomerB]
        stoichiometry = [1, -1, -1]
        reference = reaction.reference
        dict_[f"reaction-{dataset}"][reaction.dimer] = {
            "systems": systems,
            "stoichiometry": stoichiometry,
            "reference": reference,
        }

        if dataset != "NCIBLIND10":
            distance_str = f"reaction-{dataset}-{reaction.dimer.split('-')[-1]}"
            if distance_str not in dict_:
                dict_[distance_str] = {}
            dict_[distance_str][reaction.dimer] = {
                "systems": systems,
                "stoichiometry": stoichiometry,
                "reference": reference,
            }

Processing S22x5
Processing S66x8
Processing NCIBLIND10


In [3]:
dict_new = {}
for key in dict_:
    if key.startswith("molecule"):
        if dict_[key]:
            dict_new[key] = dict_[key]
    else:
        dict_new[key] = dict_[key]

with open(f"dft-fitset-{BASIS}.json", "w") as f:
    json.dump(dict_new, f)